In [ ]:
# Read in the dataset

import pandas as pd
import numpy as np

df = pd.read_parquet("modelling_dataset.parquet", engine="fastparquet")
df_filtered = df[(df['Zone_Int_ID'] == 0) & (df['Day_of_Week'] == 1) & (df['Month'] == 6) & (df['Year'] == 2016) &  (df['Time_Stamp'].dt.date.astype(str) == '2016-06-21')].sort_values(by='Hour')

df_filtered

In [ ]:
df.info()

In [ ]:
print(df.columns)


In [ ]:
import pandas as pd
import numpy as np
import statsmodels.api as sm 
from sklearn.preprocessing import StandardScaler

# Sort by time
df = df[df['Time_Stamp'].dt.year >= 2022].copy()
df = df.sort_values('Time_Stamp').reset_index(drop=True)

# Find index thresholds for 70% and 85%
n = len(df)
train_end = int(n * 0.70)
val_end = int(n * 0.85)

# Create the splits
train_df = df.iloc[:train_end]
val_df = df.iloc[train_end:val_end]
test_df = df.iloc[val_end:]

print(f"Train Period:    {train_df['Time_Stamp'].min()} to {train_df['Time_Stamp'].max()}")
print(f"Validation Period:    {val_df['Time_Stamp'].min()} to {val_df['Time_Stamp'].max()}")
print(f"Test Period:    {test_df['Time_Stamp'].min()} to {test_df['Time_Stamp'].max()}")

feature_cols = ['Year', 'Hour', 'Day_of_Week', 'Month', 'Weekend',
    'Holiday', 'Zone_Int_ID', 'Amenity', 'Crossing', 'Give_Way', 'Junction',
    'Railway', 'Station', 'Stop', 'Traffic_Signal', 'Temperature(F)', 
    'Humidity(%)', 'Pressure(in)', 'Visibility(mi)', 'Wind_Speed(mph)', 
    'Precipitation(in)', 'Weather_Clear', 'Weather_Cloudy', 'Weather_Dust/Windy',
    'Weather_Rain/Drizzle', 'Weather_Snow/Ice', 'Weather_Stormy',
    'Weather_Visibility Issues'
]

X_train, y_train = train_df[feature_cols].astype(float), train_df['Accident_Count']
X_val, y_val = val_df[feature_cols].astype(float), val_df['Accident_Count']
X_test, y_test = val_df[feature_cols].astype(float), val_df['Accident_Count']

# Scaling - Fitting only on train
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_val_scaled = scaler.transform(X_val)
X_test_scaled = scaler.transform(X_test)

# Intercept
X_train_const = sm.add_constant(X_train_scaled, has_constant='add')
X_val_const   = sm.add_constant(X_val_scaled, has_constant='add')
X_test_const  = sm.add_constant(X_test_scaled, has_constant='add')

# Fit model
model = sm.GLM(y_train, X_train_const, family=sm.families.NegativeBinomial(alpha=0.1))
results = model.fit()

# Check for Overfitting
train_pred = results.predict(X_train_const)
val_pred   = results.predict(X_val_const)

print(f"Train Mean Absolute Error: {np.mean(np.abs(y_train - train_pred)):.5f}")
print(f"Val Mean Absolute Error:   {np.mean(np.abs(y_val - val_pred)):.5f}")

print(results.summary())